In [0]:
--truncate table gold.fact_sale
MERGE INTO gold.fact_sale tgt
USING (

WITH base AS (
    SELECT
        sd.sls_ord_num,
        sd.sls_prd_key,
        sd.sls_cust_id,
        sd.sls_order_dt,
        sd.sls_ship_dt,
        sd.sls_due_dt,
        sd.sls_sales,
        sd.sls_quantity,
        sd.sls_price,
        sd.ingestion_date,
        pr.product_key
    FROM silver.crm_sales_details sd

    LEFT JOIN gold.dim_product pr
        ON sd.sls_prd_key = pr.product_number
        --AND sd.sls_order_dt >= pr.start_date
        --AND (pr.end_date IS NULL OR sd.sls_order_dt <= pr.end_date)
),

cust AS (
    SELECT
        b.*,
        cu.customer_key
    FROM base b

    LEFT JOIN gold.dim_customer cu
        ON b.sls_cust_id = cu.customer_id
),

dedup AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY sls_ord_num, sls_prd_key
            ORDER BY ingestion_date DESC
        ) AS rn
    FROM cust
)

SELECT
    sls_ord_num AS order_number,
    product_key,
    customer_key,
    sls_order_dt AS order_date,
    sls_ship_dt AS shipping_date,
    sls_due_dt AS due_date,
    sls_sales AS sales_amount,
    sls_quantity AS quantity,
    sls_price AS price,
    CURRENT_TIMESTAMP() AS created_at--,
   -- CURRENT_TIMESTAMP() AS updated_at
FROM dedup
WHERE rn = 1

) src

ON tgt.order_number = src.order_number
AND tgt.product_key = src.product_key

WHEN MATCHED AND (
    tgt.customer_key <> src.customer_key OR
    tgt.order_date <> src.order_date OR
    tgt.shipping_date <> src.shipping_date OR
    tgt.due_date <> src.due_date OR
    tgt.sales_amount <> src.sales_amount OR
    tgt.quantity <> src.quantity OR
    tgt.price <> src.price
)
THEN UPDATE SET
    tgt.customer_key = src.customer_key,
    tgt.order_date = src.order_date,
    tgt.shipping_date = src.shipping_date,
    tgt.due_date = src.due_date,
    tgt.sales_amount = src.sales_amount,
    tgt.quantity = src.quantity,
    tgt.price = src.price


WHEN NOT MATCHED THEN INSERT *
;